In [2]:
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType,
    BooleanType,
)

#1. Khởi tạo SparkSession
spark = SparkSession.builder \
    .master("spark://mindx-master:7077") \
    .appName("Lesson01_Spark") \
    .getOrCreate()


# Kiểm tra Spark đã chạy
print("=== THÔNG TIN SPARK APPLICATION ===")
print(f"Spark Version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print(f"Application Name: {spark.sparkContext.appName}")
print(f"Spark Master URL: {spark.sparkContext.master}")
print(f"Spark UI Web URL: {spark.sparkContext.uiWebUrl}")

print("Đã khởi tạo SparkSession thành công!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/24 06:22:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


=== THÔNG TIN SPARK APPLICATION ===
Spark Version: 3.5.1
Application ID: app-20260424062214-0000
Application Name: Lesson01_Spark
Spark Master URL: spark://mindx-master:7077
Spark UI Web URL: http://6c6a85b1cc55:4040
Đã khởi tạo SparkSession thành công!


In [4]:
file_path = "/opt/spark-data/construction_projects.csv"

#Đọc dữ liệu csv
df = spark.read.csv(
    path = file_path, 
    header = True, 
    inferSchema = True
)
df.show(10)

+----------+-------------------+----------+------------+----------+-----------------+---------------+------------+------------+---------------+---------------+--------------+-------------+----------------+-------------+----------+--------------+----------+-----------------+-----------+
|Project_ID|       Project_Name|    Region|Project_Type|Start_Date|Expected_End_Date|Actual_End_Date|Total_Budget|Spent_Budget|Main_Contractor|Project_Manager|Current_Status|Total_Workers|Safety_Incidents|Material_Cost|Labor_Cost|Equipment_Cost|Risk_Level|Is_Green_Building|Client_Name|
+----------+-------------------+----------+------------+----------+-----------------+---------------+------------+------------+---------------+---------------+--------------+-------------+----------------+-------------+----------+--------------+----------+-----------------+-----------+
|  PRJ-0001|Dự án Công nghiệp 1|Miền Trung| Công nghiệp| 9/21/2020|         5/9/2022|      4/20/2022|      406.63|      406.63|         Ric

26/04/24 06:48:28 WARN HeartbeatReceiver: Removing executor 1 with no recent heartbeats: 1460523 ms exceeds timeout 120000 ms
26/04/24 06:48:28 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 1460782 ms exceeds timeout 120000 ms
26/04/24 06:48:28 ERROR TaskSchedulerImpl: Lost executor 1 on 172.20.0.3: Executor heartbeat timed out after 1460523 ms
26/04/24 06:48:28 ERROR TaskSchedulerImpl: Lost executor 0 on 172.20.0.4: Executor heartbeat timed out after 1460782 ms
26/04/24 06:54:28 WARN HeartbeatReceiver: Removing executor 3 with no recent heartbeats: 1460678 ms exceeds timeout 120000 ms
26/04/24 06:54:28 ERROR TaskSchedulerImpl: Lost executor 3 on 172.20.0.4: Executor heartbeat timed out after 1460678 ms
26/04/24 06:55:40 ERROR TaskSchedulerImpl: Lost executor 2 on 172.20.0.3: worker lost: Not receiving heartbeat for 60 seconds
26/04/24 06:55:40 ERROR TaskSchedulerImpl: Lost executor 4 on 172.20.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/04/24 0

In [5]:

df.createOrReplaceTempView("my_view")

In [10]:
spark.sql(
    '''
    select Project_ID , Project_Name from my_view
    where Material_Cost + Labor_Cost + Equipment_Cost  >= Spent_Budget
    '''
).show()

+----------+------------+
|Project_ID|Project_Name|
+----------+------------+
+----------+------------+



In [13]:
spark.sql(
    '''
    select Project_Type , sum(Labor_Cost) as chi_phi_nhan_cong , sum(Material_Cost) as chi_phi_vat_lieu from my_view
    group by Project_Type
    '''
).show()

+------------+------------------+------------------+
|Project_Type| chi_phi_nhan_cong|  chi_phi_vat_lieu|
+------------+------------------+------------------+
|  Thương mại|22159.209999999992| 39555.33000000002|
| Công nghiệp| 27181.96999999999|49516.009999999995|
|    Dân dụng|25147.180000000008| 44484.48000000002|
|     Hạ tầng|23439.449999999997| 43465.54999999997|
+------------+------------------+------------------+



26/04/24 07:12:40 ERROR TaskSchedulerImpl: Lost executor 7 on 172.20.0.3: worker lost: Not receiving heartbeat for 60 seconds
26/04/24 07:12:40 ERROR TaskSchedulerImpl: Lost executor 9 on 172.20.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/04/24 07:19:28 WARN HeartbeatReceiver: Removing executor 11 with no recent heartbeats: 1461264 ms exceeds timeout 120000 ms
26/04/24 07:19:28 ERROR TaskSchedulerImpl: Lost executor 11 on 172.20.0.4: Executor heartbeat timed out after 1461264 ms


In [33]:
spark.sql(
    '''
    
    select Region , sum(Spent_Budget) as Spent_Budget,
    ROW_NUMBER() OVER (ORDER BY sum(Spent_Budget) desc ) as rn
    from my_view
    group by Region 
    '''
).show()

26/04/24 07:29:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 07:29:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 07:29:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+------------------+---+
|    Region|      Spent_Budget| rn|
+----------+------------------+---+
|  Miền Bắc| 141133.6099999999|  1|
|Miền Trung|127328.21000000008|  2|
|  Miền Nam|126164.81000000004|  3|
+----------+------------------+---+



26/04/24 07:29:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 07:29:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 07:29:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 08:08:28 WARN HeartbeatReceiver: Removing executor 27 with no recent heartbeats: 1458596 ms exceeds timeout 120000 ms
26/04/24 08:08:28 ERROR TaskSchedulerImpl: Lost executor 27 on 172.20.0.3: Executor heartbeat timed out after 1458596 ms
26/04/24 08:10:28 WARN HeartbeatReceiver: Removing executor 29 with no recent heartbeats: 1455431 ms exceeds timeout 120000 ms
26/04/24 08:10:28 WARN HeartbeatReceiver: Removing executor 28 with no recent heartbeats: 1454741 ms exceeds timeout 120000 ms
26/

In [42]:
spark.sql(
    '''
    
    select Project_Type , sum(Spent_Budget) as Spent_Budget,
    ABS(COALESCE(sum(Spent_Budget) - LAG(sum(Spent_Budget)) OVER (ORDER BY sum(Spent_Budget) desc),0)) AS prev_revenue
    from my_view
    group by Project_Type 
    '''
).show()

26/04/24 08:07:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 08:07:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 08:07:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 08:07:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 08:07:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 08:07:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 0

+------------+------------------+-----------------+
|Project_Type|      Spent_Budget|     prev_revenue|
+------------+------------------+-----------------+
| Công nghiệp|110567.83000000005|              0.0|
|    Dân dụng|          99483.62|11084.21000000005|
|     Hạ tầng| 96246.85000000005|3236.769999999946|
|  Thương mại|          88328.33|7918.520000000048|
+------------+------------------+-----------------+

